# Shape demo: load the demo Parquet files into Delta tables

Run once after uploading the demo data (produced by `demo/make_data.py`) to the
lakehouse `Files/demo/` folder. Every `*.parquet` file (one folder level down is also
accepted) becomes a Delta table named after the file, for example
`Files/demo/day1/orders.parquet` becomes the table `orders_day1`, and
`Files/demo/orders.parquet` becomes `orders`.

Attach the default lakehouse before running. Uses the preinstalled `deltalake` package.


In [ ]:
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import write_deltalake

LAKEHOUSE = "/lakehouse/default"
DEMO_DIR = Path(LAKEHOUSE) / "Files" / "demo"
TABLES_DIR = Path(LAKEHOUSE) / "Tables"
TABLES_URI = ""  # optional: abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>/Tables


In [ ]:
from urllib.parse import quote


def tables_location():
    """(where the tables go, delta-rs storage options) for this run."""
    try:
        import notebookutils
    except ImportError:
        return str(TABLES_DIR), None
    ctx = dict(getattr(getattr(notebookutils, "runtime", None), "context", None) or {})
    uri = TABLES_URI.strip().rstrip("/")
    if not uri:
        lakehouse = ctx.get("defaultLakehouseId")
        workspace = ctx.get("defaultLakehouseWorkspaceId") or ctx.get("currentWorkspaceId")
        lakehouse_name = ctx.get("defaultLakehouseName")
        workspace_name = ctx.get("defaultLakehouseWorkspaceName") or ctx.get("currentWorkspaceName")
        if lakehouse and workspace:
            uri = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Tables"
        elif lakehouse_name and workspace_name:
            uri = (
                f"abfss://{quote(workspace_name, safe='')}@onelake.dfs.fabric.microsoft.com/"
                f"{quote(lakehouse_name, safe='')}.Lakehouse/Tables"
            )
    if not hasattr(notebookutils, "credentials"):
        return str(TABLES_DIR), None
    if not uri:  # in Fabric: the mount would fail ("Unable to rename file"), so do not use it
        raise RuntimeError(
            "No default lakehouse in notebookutils.runtime.context: attach the default "
            "lakehouse, or set TABLES_URI to its Tables ABFS path"
        )
    token = notebookutils.credentials.getToken("storage")
    return uri, {"bearer_token": token, "use_fabric_endpoint": "true"}


def delta_ready(table: pa.Table) -> pa.Table:
    """The types Delta stores everywhere (as Shape's Delta sink writes them): nanosecond
    timestamps become microseconds, time-zone-less timestamps UTC, large strings strings."""
    fields = []
    for field in table.schema:
        ftype = field.type
        if pa.types.is_timestamp(ftype):
            ftype = pa.timestamp("us" if ftype.unit == "ns" else ftype.unit, tz=ftype.tz or "UTC")
        elif pa.types.is_large_string(ftype):
            ftype = pa.string()
        fields.append(pa.field(field.name, ftype, field.nullable))
    return table.cast(pa.schema(fields), safe=False)


TABLES, STORAGE_OPTIONS = tables_location()
print("Writing Delta tables to", TABLES)


In [ ]:
def table_name_for(parquet_file: Path) -> str:
    """<sub>/<n>.parquet -> <n>_<sub>;  <n>.parquet -> <n>."""
    rel = parquet_file.relative_to(DEMO_DIR)
    parts = [p.lower().replace("-", "_").replace(" ", "_") for p in rel.parts]
    stem = Path(parts[-1]).stem
    return "_".join([stem, *parts[:-1]]) if len(parts) > 1 else stem


files = sorted(DEMO_DIR.rglob("*.parquet"))
if not files:
    raise FileNotFoundError(
        f"No .parquet files under {DEMO_DIR}. Upload the demo data to Files/demo/ first."
    )


In [ ]:
created = []
for f in files:
    name = table_name_for(f)
    table = delta_ready(pq.read_table(f))
    write_deltalake(
        f"{TABLES}/{name}",
        table,
        mode="overwrite",
        schema_mode="overwrite",
        storage_options=STORAGE_OPTIONS,
    )
    created.append({"table": name, "rows": table.num_rows, "source": str(f.relative_to(DEMO_DIR))})

for row in created:
    print(f"{row['table']:<32} {row['rows']:>10,} rows   <- {row['source']}")
